# Career Market Intelligence — Data Preprocessing Pipeline
**Owner / Responsibilty:** Member 1 (Data Preprocessing)  
**Project Goal:** Transform the raw, consolidated job market dataset (`data/final/career_market_core_analysis.csv`) into a standardized, analysis-ready table (`data/processed/career_market_preprocessed.csv`) for the downstream team.  
**Target Variable:** `salary_band` (`low` / `medium` / `high`) — India market classification scope.  

---

### Pipeline Architecture & Downstream Hand-off
```
Member 1: Preprocessing   <-- THIS NOTEBOOK (produces clean table)
Member 2: EDA & Visualization
Member 3 & 4: Feature Engineering & Predictive Modeling
Member 5: Evaluation & Business Recommendations
```

### Preprocessing Checklist & Scope
1. **Load Raw Dataset:** Read all 7,184 postings from `data/final/career_market_core_analysis.csv`.
2. **Column Selection:** Keep core analytical & modeling columns, drop 33 audit/provenance metadata fields.
3. **Target Variable Construction:** Calculate `salary_mid`, filter valid rows via `salary_passes_plausibility_screen`, generate balanced tertiles (`low`, `medium`, `high`), and set `has_salary` flag.
4. **Seniority Level Extraction:** Rule-based heuristic deriving `Entry`, `Mid`, `Senior` from job titles.
5. **Missing Value Treatment:** Impute structural missingness (`contract_time`, `location_region`, `employer` as `'unknown'`), strictly preserve target missingness without synthetic imputation, and drop empty descriptions.
6. **Text Cleaning & Lemmatization:** Strip HTML/URLs/emails, protect technical keywords (`c++`, `.net`, `node.js`), remove non-negation stopwords, and lemmatize using spaCy (`en_core_web_sm`).
7. **Standardization:** Datetime conversion, casing normalization, and deduplication verification.
8. **Export & Validation:** Output verified dataset to `data/processed/career_market_preprocessed.csv`.


In [ ]:
import re
import warnings
import pandas as pd
import numpy as np
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 100)

print("Environment setup completed.")

Environment setup completed.


## Step 1: Load Raw Dataset
We load `data/final/career_market_core_analysis.csv`, which contains 7,184 rows across 49 columns.

In [ ]:
INPUT_PATH = "../data/final/career_market_core_analysis.csv"
OUTPUT_PATH = "../data/processed/career_market_preprocessed.csv"

df_raw = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"Dataset Loaded Successfully: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head(3)

Dataset Loaded Successfully: 7,184 rows x 49 columns


## Step 2: Feature Selection (Drop Audit-Only Columns)
We retain the 16 analytical features and drop 33 columns that were used exclusively for collection provenance, deduplication audits, and scrapers.

In [ ]:
KEEP_COLS = [
    "job_id", "title", "role", "domain", "employer",
    "description_raw", "location_country", "location_region",
    "created_date", "category_label", "contract_time",
    "salary_min", "salary_max", "salary_present",
    "salary_passes_plausibility_screen", "description_length",
]

df = df_raw[KEEP_COLS].copy()
print(f"Retained {len(KEEP_COLS)} columns. Dropped {df_raw.shape[1] - len(KEEP_COLS)} audit columns.")
df.info()

Retained 16 columns. Dropped 33 audit columns.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7184 entries, 0 to 7183
Data columns (total 16 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   job_id                             7184 non-null   int64  
 1   title                              7184 non-null   object 
 2   role                               7184 non-null   object 
 3   domain                             7184 non-null   object 
 4   employer                           6912 non-null   object 
 5   description_raw                    7184 non-null   object 
 6   location_country                   7184 non-null   object 
 7   location_region                    4736 non-null   object 
 8   created_date                       7184 non-null   object 
 9   category_label                     7184 non-null   object 
 10  contract_time                      4308 non-null   object 
 11  salary_mi

## Step 3: Build the Target Variable (`salary_band`)
The classification target is **`salary_band`** (`low`, `medium`, `high`) in Indian Rupees (INR).
* **Midpoint calculation:** `salary_mid = df[['salary_min', 'salary_max']].mean(axis=1)`
* **Plausibility filter:** Salary values are restricted strictly to rows where `salary_passes_plausibility_screen == True` (1,101 rows).
* **Tertile discretization:** `pd.qcut(df.loc[has_salary, 'salary_mid'], 3, labels=['low', 'medium', 'high'])`.
* **Important rule:** Never invent or impute missing salaries for the remaining postings — they are retained for EDA and NLP tasks with `has_salary = False`.

In [ ]:
# 1. Compute midpoint (handles both ranges and fixed salaries)
df["salary_mid"] = df[["salary_min", "salary_max"]].mean(axis=1)

# 2. Invalidate non-plausible salaries
df.loc[~df["salary_passes_plausibility_screen"], "salary_mid"] = np.nan

# 3. Indicator flag for supervised modeling
df["has_salary"] = df["salary_passes_plausibility_screen"].fillna(False)

# 4. Construct 3-class target via tertiles on plausible rows
plausible_mask = df["has_salary"]
df["salary_band"] = pd.NA
df.loc[plausible_mask, "salary_band"] = pd.qcut(
    df.loc[plausible_mask, "salary_mid"],
    q=3,
    labels=["low", "medium", "high"]
)

print(f"Total postings with valid target (has_salary == True): {df['has_salary'].sum():,}")
print("\nTarget Class Distribution (salary_band):")
print(df["salary_band"].value_counts(dropna=False))

# Examine tertile cut-off points
cuts = pd.qcut(df.loc[plausible_mask, "salary_mid"], q=3).value_counts().sort_index()
print("\nTertile Boundary Intervals (INR):")
for interval, cnt in cuts.items():
    print(f"  {interval}: {cnt} rows")

Total postings with valid target (has_salary == True): 1,101

Target Class Distribution (salary_band):
salary_band
<NA>      6083
low        401
high       363
medium     337
Name: count, dtype: int64

Tertile Boundary Intervals (INR):
  (99999.999, 600000.0]: 401 rows
  (600000.0, 1600000.0]: 337 rows
  (1600000.0, 5000000.0]: 363 rows


## Step 4: Derive `seniority_level` Feature from Job Title
We extract seniority as a predictive feature using regex keyword patterns:
* **Senior:** `senior`, `sr`, `lead`, `principal`, `head`, `manager`, `director`
* **Entry:** `junior`, `jr`, `graduate`, `trainee`, `intern`, `entry`, `associate`
* **Mid:** default for all remaining positions

In [ ]:
SENIOR_KEYWORDS = r"\b(senior|sr\.?|lead|principal|head|manager|director)\b"
ENTRY_KEYWORDS  = r"\b(junior|jr\.?|graduate|trainee|intern|entry|associate)\b"

def derive_seniority(title: str) -> str:
    if not isinstance(title, str):
        return "Mid"
    t = title.lower()
    if re.search(SENIOR_KEYWORDS, t):
        return "Senior"
    if re.search(ENTRY_KEYWORDS, t):
        return "Entry"
    return "Mid"

df["seniority_level"] = df["title"].apply(derive_seniority)

print("Seniority Level Distribution:")
print(df["seniority_level"].value_counts())
print("\nSeniority Level vs Target (has_salary == True):")
print(pd.crosstab(df["seniority_level"], df["salary_band"], margins=True))

Seniority Level Distribution:
seniority_level
Mid       4010
Senior    2975
Entry      199
Name: count, dtype: int64

Seniority Level vs Target (has_salary == True):
salary_band      high  low  medium   All
seniority_level                         
Entry               0   32       7    39
Mid               170  234     200   604
Senior            193  135     130   458
All               363  401     337  1101


## Step 5: Handle Missing Values & Audit Logging
* `contract_time` (~40% missing) -> filled with `"unknown"`
* `location_region` (~34% missing) -> filled with `"unknown"`
* `employer` (~3.8% missing) -> filled with `"unknown"`
* `salary_min`, `salary_max` -> untouched (target integrity preserved)
* Empty `description_raw` -> verified and dropped if present.

In [ ]:
TRACK_COLS = ["salary_min", "salary_max", "contract_time", "location_region", "employer", "description_raw"]

print("=== MISSING VALUES BEFORE CLEANING ===")
for col in TRACK_COLS:
    cnt = df[col].isna().sum()
    pct = (cnt / len(df)) * 100
    print(f"  {col:<20}: {cnt:4d} missing ({pct:5.2f}%)")

# Impute unknown categorical labels
df["contract_time"]   = df["contract_time"].fillna("unknown")
df["location_region"] = df["location_region"].fillna("unknown")
df["employer"]        = df["employer"].fillna("unknown")

# Verify description_raw emptiness
empty_desc = df["description_raw"].isna() | (df["description_raw"].str.strip() == "")
if empty_desc.sum() > 0:
    print(f"\nDropping {empty_desc.sum()} rows with empty description_raw.")
    df = df[~empty_desc].reset_index(drop=True)
else:
    print("\nZero rows with empty description_raw.")

print("\n=== MISSING VALUES AFTER CLEANING ===")
for col in TRACK_COLS:
    cnt = df[col].isna().sum()
    pct = (cnt / len(df)) * 100
    print(f"  {col:<20}: {cnt:4d} missing ({pct:5.2f}%)")

=== MISSING VALUES BEFORE CLEANING ===
  salary_min          : 5931 missing (82.56%)
  salary_max          : 5934 missing (82.60%)
  contract_time       : 2710 missing (37.72%)
  location_region     : 2406 missing (33.49%)
  employer            :    1 missing ( 0.01%)
  description_raw     :    0 missing ( 0.00%)

Zero rows with empty description_raw.

=== MISSING VALUES AFTER CLEANING ===
  salary_min          : 5931 missing (82.56%)
  salary_max          : 5934 missing (82.60%)
  contract_time       :    0 missing ( 0.00%)
  location_region     :    0 missing ( 0.00%)
  employer            :    0 missing ( 0.00%)
  description_raw     :    0 missing ( 0.00%)


## Step 6: Text Cleaning & Normalization (`description_raw` -> `description_clean`)
We process the job descriptions using an NLP pipeline:
1. Strip HTML tags and decode HTML entities (`&amp;`, `&lt;`, etc.).
2. Remove URLs and email addresses.
3. Protect tech tokens (e.g. `c++`, `c#`, `.net`, `node.js`, `react.js`).
4. Strip non-alphanumeric punctuation.
5. SpaCy lemmatization (`en_core_web_sm`) and stopword removal (**preserving negation words** such as `not`, `no`, `without`, `cannot`).
6. Restore tech tokens and collapse redundant whitespace.

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

NEGATIONS = {
    "not", "no", "nor", "neither", "never", "none", "nothing",
    "nowhere", "nobody", "without", "cannot",
}
STOP_WORDS_FILTERED = STOP_WORDS - NEGATIONS

TECH_TOKENS = {
    "c++":      "__CPLUSPLUS__",
    "c#":       "__CSHARP__",
    ".net":     "__DOTNET__",
    "node.js":  "__NODEJS__",
    "vue.js":   "__VUEJS__",
    "react.js": "__REACTJS__",
    "asp.net":  "__ASPNET__",
}
TECH_RESTORE = {v.lower(): k for k, v in TECH_TOKENS.items()}

def protect_tech_tokens(text: str) -> str:
    for token, placeholder in TECH_TOKENS.items():
        text = text.replace(token, placeholder)
    return text

def restore_tech_tokens(text: str) -> str:
    for placeholder, token in TECH_RESTORE.items():
        text = text.replace(placeholder, token)
    return text

def clean_description(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    # 1. HTML stripping
    text = re.sub(r"<[^>]+>", " ", text)
    for ent, repl in [("&amp;","&"),("&lt;","<"),("&gt;",">"),("&nbsp;"," "),("&quot;",'"'),("&#39;","'")]:
        text = text.replace(ent, repl)
    # 2. URLs and emails
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+\.\S+", " ", text)
    # 3. Protect tech tokens & lower
    text = protect_tech_tokens(text.lower())
    # 4. Filter characters
    text = re.sub(r"[^a-z0-9 _]", " ", text)
    # 5. Lemmatize & remove non-negation stopwords
    doc = nlp(text)
    tokens = [
        token.lemma_
        for token in doc
        if token.text not in STOP_WORDS_FILTERED
        and token.lemma_ not in STOP_WORDS_FILTERED
        and not token.is_space
        and token.text.strip()
    ]
    text = " ".join(tokens)
    # 6. Restore tech tokens
    text = restore_tech_tokens(text)
    return re.sub(r"\s+", " ", text).strip()

print("Applying NLP cleaning pipeline on description_raw...")
df["description_clean"] = df["description_raw"].apply(clean_description)
print("Text cleaning complete.")

print("\nSample Transformation (Row 0):")
print(f"RAW  : {df['description_raw'].iloc[0][:150]}...")
print(f"CLEAN: {df['description_clean'].iloc[0][:150]}...")

Applying NLP cleaning pipeline on description_raw...
Text cleaning complete.

Sample Transformation (Row 0):
RAW  : Be an essential element to a brighter future. We work together to transform essential resources into critical ingredient...
CLEAN: essential element bright future work transform essential resource critical ingredient mobility energy connectivity healt...


## Step 7: Standardization & Integrity Checks
* Convert `created_date` to pandas `datetime64[ns]`.
* Standardize string columns to `.str.strip().str.title()` (`role`, `location_region`, `location_country`, `category_label`).
* Verify uniqueness of primary key (`job_id`).

In [ ]:
# Datetime conversion
df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")

# String standardization
STR_COLS = ["role", "location_region", "location_country", "category_label"]
for col in STR_COLS:
    df[col] = df[col].astype(str).str.strip().str.title()

# Deduplication / uniqueness check
dup_count = df["job_id"].duplicated().sum()
assert dup_count == 0, f"Duplicate job_ids detected: {dup_count}"
print(f"Uniqueness check passed: {len(df):,} unique job_ids (0 duplicates).")
print(f"created_date range: {df['created_date'].min().date()} to {df['created_date'].max().date()}")

Uniqueness check passed: 7,184 unique job_ids (0 duplicates).
created_date range: 2026-01-02 to 2026-03-31


## Step 8: Final Validation, Schema Verification & Export
We arrange the final 21 columns and export to `data/processed/career_market_preprocessed.csv`.

In [ ]:
FINAL_COLS = KEEP_COLS + [
    "salary_mid", "salary_band", "has_salary",
    "seniority_level", "description_clean"
]

df_out = df[FINAL_COLS].copy()

# Validation checks
assert df_out.shape[0] == 7184, f"Row count mismatch: {df_out.shape[0]}"
assert df_out.shape[1] == 21, f"Column count mismatch: {df_out.shape[1]}"
assert df_out["job_id"].is_unique, "job_id is not unique"
assert df_out["has_salary"].sum() == 1101, f"Expected 1,101 salaried rows, got {df_out['has_salary'].sum()}"

# Export to CSV
df_out.to_csv(OUTPUT_PATH, index=False)
print(f"Preprocessed dataset saved to: {OUTPUT_PATH}")
print(f"Final Table Shape: {df_out.shape[0]:,} rows x {df_out.shape[1]} columns")

print("\n=== DEFINITION OF DONE VERIFICATION ===")
print(f" [x] career_market_preprocessed.csv exported: Yes")
print(f" [x] 7,184 rows preserved & job_id unique:    Yes ({len(df_out):,} rows)")
print(f" [x] salary_band built on plausible rows:      Yes ({df_out['has_salary'].sum():,} rows)")
print(f" [x] has_salary flag present:                   Yes (True: {df_out['has_salary'].sum()}, False: {(~df_out['has_salary']).sum()})")
print(f" [x] seniority_level derived:                   Yes (Entry: {(df_out['seniority_level']=='Entry').sum()}, Mid: {(df_out['seniority_level']=='Mid').sum()}, Senior: {(df_out['seniority_level']=='Senior').sum()})")
print(f" [x] description_clean created:                 Yes")
print(f" [x] No synthetic salary values invented:       Yes (Missing salary remains NaN)")

Preprocessed dataset saved to: ../data/processed/career_market_preprocessed.csv
Final Table Shape: 7,184 rows x 21 columns

=== DEFINITION OF DONE VERIFICATION ===
 [x] career_market_preprocessed.csv exported: Yes
 [x] 7,184 rows preserved & job_id unique:    Yes (7,184 rows)
 [x] salary_band built on plausible rows:      Yes (1,101 rows)
 [x] has_salary flag present:                   Yes (True: 1101, False: 6083)
 [x] seniority_level derived:                   Yes (Entry: 199, Mid: 4010, Senior: 2975)
 [x] description_clean created:                 Yes
 [x] No synthetic salary values invented:       Yes (Missing salary remains NaN)


## Hand-off Summary for Team Members
* **To Member 2 (EDA):** The clean table is available at `data/processed/career_market_preprocessed.csv`. All 7,184 rows are available for market distribution, role trends, and location analysis.
* **To Members 3 & 4 (Feature Engineering & Modeling):**
  * Train the classifier only on rows where `has_salary == True` (1,101 rows).
  * Target variable: `salary_band` (`low`, `medium`, `high`).
  * Features available: `role`, `domain`, `employer`, `location_region`, `contract_time`, `seniority_level`, and `description_clean`.
